In [71]:
import pandas as pd
import os
from config import data_inicial, data_final

tamanho = os.path.getsize("../data/inf_diario_fi_202609.csv")
print(tamanho)

tabela = pd.read_csv("../data/inf_diario_fi_202609.csv", sep=";")

print(tabela)

df = pd.DataFrame(tabela)

id_subclasse_nulo = df['ID_SUBCLASSE'].isna().sum()

print(id_subclasse_nulo)

print(df.memory_usage(deep=True).sum())

30649745
       TP_FUNDO_CLASSE   CNPJ_FUNDO_CLASSE ID_SUBCLASSE   DT_COMPTC  \
0        CLASSES - FIF  00.017.024/0001-53          NaN  2026-09-01   
1        CLASSES - FIF  00.017.024/0001-53          NaN  2026-09-02   
2        CLASSES - FIF  00.017.024/0001-53          NaN  2026-09-03   
3        CLASSES - FIF  00.017.024/0001-53          NaN  2026-09-04   
4        CLASSES - FIF  00.017.024/0001-53          NaN  2026-09-08   
...                ...                 ...          ...         ...   
302840   CLASSES - FIF  97.929.213/0001-34          NaN  2026-09-11   
302841   CLASSES - FIF  97.929.213/0001-34          NaN  2026-09-14   
302842   CLASSES - FIF  97.929.213/0001-34          NaN  2026-09-15   
302843   CLASSES - FIF  97.929.213/0001-34          NaN  2026-09-16   
302844   CLASSES - FIF  97.929.213/0001-34          NaN  2026-09-17   

           VL_TOTAL   VL_QUOTA  VL_PATRIM_LIQ  CAPTC_DIA  RESG_DIA  NR_COTST  
0        1107741.39  44.636723     1214792.63        0.0   

In [72]:
fundo = df[df["CNPJ_FUNDO_CLASSE"] == "00.068.305/0001-35"].sort_values(by="DT_COMPTC")

fundo["PL_DIFF"] = fundo["VL_PATRIM_LIQ"].diff()
fundo["CAPTC_RESG"] = fundo["CAPTC_DIA"] - fundo["RESG_DIA"]
fundo["ERRO"] = fundo["PL_DIFF"] - fundo["CAPTC_RESG"]

fundo_formatado = fundo[["DT_COMPTC", "VL_PATRIM_LIQ", "PL_DIFF", "CAPTC_DIA", "RESG_DIA", "CAPTC_RESG", "ERRO"]].head(10)
print(fundo_formatado)
# Estatisticas descritivas do numero de cotistas no período
fundo["NR_COTST"].describe()

     DT_COMPTC  VL_PATRIM_LIQ   PL_DIFF  CAPTC_DIA  RESG_DIA  CAPTC_RESG  \
12  2026-09-01    41321491.71       NaN        0.0    100.47     -100.47   
13  2026-09-02    41340449.99  18958.28        0.0      0.00        0.00   
14  2026-09-03    41358485.07  18035.08        0.0      0.00        0.00   
15  2026-09-04    41366372.17   7887.10        0.0   9956.40    -9956.40   
16  2026-09-08    41384888.97  18516.80        0.0    181.67     -181.67   
17  2026-09-09    41402341.79  17452.82        0.0   1478.61    -1478.61   
18  2026-09-10    41418027.60  15685.81        0.0   2713.94    -2713.94   
19  2026-09-11    41436938.63  18911.03        0.0    603.12     -603.12   
20  2026-09-14    41455434.40  18495.77        0.0     81.97      -81.97   
21  2026-09-15    41474790.68  19356.28        0.0      0.00        0.00   

        ERRO  
12       NaN  
13  18958.28  
14  18035.08  
15  17843.50  
16  18698.47  
17  18931.43  
18  18399.75  
19  19514.15  
20  18577.74  
21  19356.28 

count      12.000000
mean     5539.166667
std         2.124889
min      5537.000000
25%      5537.000000
50%      5538.500000
75%      5541.250000
max      5542.000000
Name: NR_COTST, dtype: float64

In [73]:
fundo["RENDIMENTO_ESTIMADO"] = fundo["VL_QUOTA"].pct_change() * fundo["VL_PATRIM_LIQ"].shift(1)

In [74]:
fundo[["DT_COMPTC", "ERRO", "RENDIMENTO_ESTIMADO"]].head(10)

,DT_COMPTC,ERRO,RENDIMENTO_ESTIMADO
12,2026-09-01,NaN,NaN
13,2026-09-02,18958.28,18957.791585
14,2026-09-03,18035.08,18035.059725
15,2026-09-04,17843.50,17843.549301
16,2026-09-08,18698.47,18698.234274
17,2026-09-09,18931.43,18932.259955
18,2026-09-10,18399.75,18398.985827
19,2026-09-11,19514.15,19514.247985
20,2026-09-14,18577.74,18577.641718
21,2026-09-15,19356.28,19356.421671


In [75]:
import requests

data_inicial = "01/01/2026"
data_final = "21/10/2026"

def buscar_serie_bcb(codigo, nome_serie, data_inicial, data_final):
    url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}'
    resposta = requests.get(url)

    if resposta.status_code != 200:
        return "ERRO, código inválido"

    dados = resposta.json()
    df = pd.DataFrame(dados)
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
    df["valor"] = df["valor"].astype(float)
    df = df.rename(columns={"valor": f"valor_{nome_serie}"})

    return df

In [76]:
# df_selic = pd.DataFrame(dados)
# df_selic["data"] = pd.to_datetime(df_selic["data"], format="%d/%m/%Y")
# df_selic["valor"] = df_selic["valor"].astype(float)
# df_selic.dtypes

In [77]:
df_selic = buscar_serie_bcb(11,"selic", data_inicial, data_final)
df_cambio = buscar_serie_bcb(10813, "cambio", data_inicial, data_final)
df_ipca = buscar_serie_bcb(433, "ipca", data_inicial, data_final)


In [78]:
df_selic.shape

(201, 2)

In [79]:
df_selic.head(5)

,data,valor_selic
0,2026-01-02,0.055131
1,2026-01-05,0.055131
2,2026-01-06,0.055131
3,2026-01-07,0.055131
4,2026-01-08,0.055131


In [80]:
df_cambio.shape

(201, 2)

In [81]:
df_cambio.head(5)

,data,valor_cambio
0,2026-01-02,5.4366
1,2026-01-05,5.4345
2,2026-01-06,5.3791
3,2026-01-07,5.3874
4,2026-01-08,5.3854


In [82]:
df_ipca.shape

(9, 2)

In [83]:
df_ipca.head()

,data,valor_ipca
0,2026-01-01,0.33
1,2026-02-01,0.70
2,2026-03-01,0.88
3,2026-04-01,0.67
4,2026-05-01,0.58


In [84]:

df_ipca["data"].max()

Timestamp('2026-09-01 00:00:00')